In [4]:
import os
from ratelimit import limits, sleep_and_retry
import requests
import json
from urllib3.util import Retry
import sqlite3

In [5]:
os.remove('test.db')

PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: 'test.db'

In [ ]:
EPOCH_TIME_JULY2026 = 1782867600
RANKED_SOLO = 420
ONE_SECOND = 1
TWO_MINUTES = 120
NUM_CHAMPIONS_PER_GAME = 10
NUM_GAMES_PER_PLAYER = 1
CURRENT_PATCH = '16.14.1' # Update this with the current patch version

csv_filename = 'champ_data.csv'

platforms = ['OC1', 'JP1', 'KR', 'BR1', 'LA1', 'LA2', 'NA1', 'TR1', 'RU', 'EUN1', 'EUW1', 'ME1', 'SG2', 'TW2', 'VN2']
regions = {'OC1':'sea', 'SG2': 'sea', 'TW2': 'sea', 'VN2': 'sea', 'JP1': 'asia', 'KR': 'asia', 'BR1': 'americas', 'LA1': 'americas', 'LA2': 'americas', 'NA1': 'americas', 'TR1': 'europe', 'RU': 'europe', 'EUN1': 'europe', 'EUW1': 'europe', 'ME1': 'europe'}

champion_names_url = 'https://ddragon.leagueoflegends.com/cdn/{version}/data/en_US/champion.json'
master_division_url = 'https://{platform}.api.riotgames.com/lol/league/v4/masterleagues/by-queue/RANKED_SOLO_5x5'
matches_by_player_url = 'https://{region}.api.riotgames.com/lol/match/v5/matches/by-puuid/{puuid}/ids?startTime={start_time}&queue={queue}&type=ranked&start=0&count={count}'
match_data_from_matchid = 'https://{region}.api.riotgames.com/lol/match/v5/matches/{matchId}'
api_key = os.getenv("RIOT_API_KEY")

headers = {
    'X-Riot-Token': api_key
}

In [30]:
session = requests.Session()
retries = Retry(total=10,
                backoff_factor=2,
                status_forcelist=[429, 500, 502, 503, 504])
session.mount('https://', requests.adapters.HTTPAdapter(max_retries=retries))

In [31]:
@sleep_and_retry
@limits(calls=95, period=TWO_MINUTES)
@limits(calls=18, period=ONE_SECOND)
def call_api(url, headers=None):
    response = session.get(url, headers=headers)

    if response.status_code >= 400:
        print(f'Status: {response.status_code} Url: {url}')
        return None
    
    return response

In [ ]:
for platform in platforms:
    player_data = call_api(master_division_url.format(platform=platform), headers)

    data = player_data.json()['entries']
    player_id = [player['puuid'] for player in data] # collects all the player puuids from the master division

    match_data = []
    for puuid in player_id:
        url = matches_by_player_url.format(region=regions[platform],
                                           puuid=puuid,
                                           start_time=EPOCH_TIME_JULY2026,
                                           queue=RANKED_SOLO,
                                           count=NUM_GAMES_PER_PLAYER)
        response = call_api(url, headers)
        
        if not response:
            continue
        match_data.extend(response.json())
        print(f'{len(player_id) * NUM_GAMES_PER_PLAYER - len(match_data)} matches left to go in {platform}')

    match_data = list(set(match_data)) # removes duplicates

    print('Match data collected!')
    champ_data = []
    for match_id in match_data:
        url = match_data_from_matchid.format(region=regions[platform],
                                             matchId=match_id)
        
        response = call_api(url, headers)

        if not response:
            continue
        
        players = response.json()['info']['participants']
        current_champs = [player['championName'] for player in players] # first 5 players are team 1, next 5 are team 2
        champ_data.append(current_champs)

        print(f'{len(match_data) - len(champ_data)} matches left to parse for {platform}')

1798 matches left to go in OC1
1797 matches left to go in OC1
1796 matches left to go in OC1
1795 matches left to go in OC1
1794 matches left to go in OC1
1793 matches left to go in OC1
1792 matches left to go in OC1
1791 matches left to go in OC1
1790 matches left to go in OC1
1789 matches left to go in OC1
1788 matches left to go in OC1
1787 matches left to go in OC1
1786 matches left to go in OC1
1785 matches left to go in OC1
1784 matches left to go in OC1
1783 matches left to go in OC1
1782 matches left to go in OC1
1781 matches left to go in OC1
1780 matches left to go in OC1
1779 matches left to go in OC1
1778 matches left to go in OC1
1777 matches left to go in OC1
1776 matches left to go in OC1
1775 matches left to go in OC1
1774 matches left to go in OC1
1773 matches left to go in OC1
1772 matches left to go in OC1
1771 matches left to go in OC1
1770 matches left to go in OC1
1769 matches left to go in OC1
1768 matches left to go in OC1
1767 matches left to go in OC1
1766 mat

KeyboardInterrupt: 

In [ ]:
all_champion_names = call_api(champion_names_url.format(version=CURRENT_PATCH))

session.close()

In [80]:
with open('champ_names.json', 'w') as f:
    json.dump(list(all_champion_names.json()['data'].keys()), f)

In [63]:
with open('champ_data.json', 'w') as f:
    json.dump(champ_data, f)